In [1]:
import sys
sys.path.append('..')   # lets the notebook import preprocess.py from the project root

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from preprocess import preprocess

df = pd.read_csv('../data/python_qa_processed.csv')
print(df.shape)

(171, 5)


In [2]:
THRESHOLD = 0.2

extra_keywords = {
    89: "create list from another list one line shortcut loop compact",
    119: "handle handle error handle errors error errors try catch bug crash",
}
df['keywords'] = df['id'].map(extra_keywords).fillna('')
df['search_text'] = (df['question'] + ' ' + df['keywords']).apply(preprocess)

vectorizer2 = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix2 = vectorizer2.fit_transform(df['search_text'])

# (question, expected id). None = should be rejected
final_tests = [
    ("how to create a list from another list in one line", 89),
    ("difference between / and //", 20),
    ("What does the // operator do?", 20),
    ("what is a lambda", 73),
    ("how do I handle errors in python", 119),
    ("how do I define a function", 67),
    ("what is a dictionary", 60),
    ("how to open and read a file", 127),
    ("how do I reverse a string", 36),
    ("what is a decorator", 93),
    ("what is the capital of France", None),
    ("who is the president of india", None),
    ("who won the world cup", None),
    ("how to cook rice", None),
    ("what is the tallest mountain", None),
]

py_ok = py_n = off_ok = off_n = 0
for q, expected in final_tests:
    vec = vectorizer2.transform([preprocess(q)])
    scores = cosine_similarity(vec, tfidf_matrix2).flatten()
    idx = scores.argmax()
    score, matched = scores[idx], df.iloc[idx]['id']
    if expected is None:
        off_n += 1
        ok = score < THRESHOLD
        off_ok += ok
    else:
        py_n += 1
        ok = (matched == expected) and (score >= THRESHOLD)
        py_ok += ok
    print(f"{'PASS' if ok else 'FAIL'} | {score:.2f} | id={matched} | {q}")

print(f"\nPython questions: {py_ok}/{py_n}")
print(f"Off-topic rejected: {off_ok}/{off_n}")


PASS | 0.72 | id=89 | how to create a list from another list in one line
PASS | 0.90 | id=20 | difference between / and //
PASS | 0.31 | id=20 | What does the // operator do?
PASS | 0.55 | id=73 | what is a lambda
PASS | 0.72 | id=119 | how do I handle errors in python
PASS | 0.64 | id=67 | how do I define a function
PASS | 0.59 | id=60 | what is a dictionary
PASS | 0.82 | id=127 | how to open and read a file
PASS | 0.85 | id=36 | how do I reverse a string
PASS | 0.64 | id=93 | what is a decorator
PASS | 0.00 | id=1 | what is the capital of France
PASS | 0.00 | id=1 | who is the president of india
PASS | 0.00 | id=1 | who won the world cup
PASS | 0.00 | id=1 | how to cook rice
PASS | 0.00 | id=1 | what is the tallest mountain

Python questions: 10/10
Off-topic rejected: 5/5


In [3]:
df.to_csv('../data/python_qa_final.csv', index=False)
print("Saved:", df.shape)
print(df.columns.tolist())

Saved: (171, 7)
['id', 'category', 'question', 'answer', 'clean_question', 'keywords', 'search_text']


In [2]:
# Create the TF-IDF vectorizer (unigrams + bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))

# Learn the vocabulary from the cleaned questions and convert them to vectors
tfidf_matrix = vectorizer.fit_transform(df['clean_question'])

print("Matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Sample terms:", list(vectorizer.get_feature_names_out()[:10]))

Matrix shape: (171, 675)
Vocabulary size: 675
Sample terms: ['abstraction', 'abstraction in', 'access', 'access dictionary', 'add', 'add update', 'and', 'and append', 'and argument', 'and array']


In [3]:
def get_answer(user_question, threshold=0.3):
    # Clean the user's question the same way we cleaned the dataset
    clean = preprocess(user_question)

    # Convert it to a vector using the SAME vectorizer
    user_vector = vectorizer.transform([clean])

    # Compare against all 171 question vectors
    scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # Find the best match
    best_idx = scores.argmax()
    best_score = scores[best_idx]

    if best_score < threshold:
        return "Sorry, I don't know the answer to that yet.", best_score

    return df.iloc[best_idx]['answer'], best_score


# Quick test
for q in ["What does the // operator do?",
          "how do I make a list comprehension",
          "what is the capital of France"]:
    answer, score = get_answer(q)
    print("Q:", q)
    print("Score:", round(score, 2))
    print("A:", answer)
    print("-" * 40)

Q: What does the // operator do?
Score: 0.31
A: / is true division and always returns a float (7 / 2 gives 3.5). // is floor division and returns the quotient rounded down (7 // 2 gives 3).
----------------------------------------
Q: how do I make a list comprehension
Score: 0.72
A: List comprehension is a short way to create a list from an iterable. Syntax: [expression for item in iterable if condition]. Example: squares = [x * x for x in range(5)] gives [0, 1, 4, 9, 16].
----------------------------------------
Q: what is the capital of France
Score: 0.0
A: Sorry, I don't know the answer to that yet.
----------------------------------------


In [4]:
# Test questions worded differently from the dataset.
# "expected" is the id of the dataset row that should be matched.
# Replace the ids below with real ids from YOUR data (see step 3).
test_cases = [
    ("how to create a list from another list in one line", 0),
    ("difference between / and //", 0),
    ("what is a lambda", 0),
    ("how do I handle errors in python", 0),
    ("what is the capital of France", None),   # None = should be rejected
]

correct = 0
for question, expected_id in test_cases:
    answer, score = get_answer(question)
    best_idx = vectorizer.transform([preprocess(question)])
    scores = cosine_similarity(best_idx, tfidf_matrix).flatten()
    matched_id = df.iloc[scores.argmax()]['id']

    if expected_id is None:
        ok = score < 0.3
    else:
        ok = (matched_id == expected_id)

    correct += ok
    print(f"{'PASS' if ok else 'FAIL'} | score={score:.2f} | matched id={matched_id} | {question}")

print(f"\nAccuracy: {correct}/{len(test_cases)}")

FAIL | score=0.46 | matched id=43 | how to create a list from another list in one line
FAIL | score=0.88 | matched id=20 | difference between / and //
FAIL | score=0.47 | matched id=73 | what is a lambda
FAIL | score=0.33 | matched id=122 | how do I handle errors in python
PASS | score=0.00 | matched id=1 | what is the capital of France

Accuracy: 1/5


In [5]:
test_questions = [
    "how to create a list from another list in one line",
    "difference between / and //",
    "what is a lambda",
    "how do I handle errors in python",
]

for q in test_questions:
    vec = vectorizer.transform([preprocess(q)])
    scores = cosine_similarity(vec, tfidf_matrix).flatten()
    idx = scores.argmax()
    print("Asked  :", q)
    print("Matched:", df.iloc[idx]['question'], f"(id={df.iloc[idx]['id']}, score={scores[idx]:.2f})")
    print("-" * 40)

Asked  : how to create a list from another list in one line
Matched: What is a list in Python? (id=43, score=0.46)
----------------------------------------
Asked  : difference between / and //
Matched: What is the difference between / and // in Python? (id=20, score=0.88)
----------------------------------------
Asked  : what is a lambda
Matched: What is a lambda function in Python? (id=73, score=0.47)
----------------------------------------
Asked  : how do I handle errors in python
Matched: How do you handle multiple exceptions in Python? (id=122, score=0.33)
----------------------------------------


In [6]:
     df[df['question'].str.contains('lambda', case=False)][['id', 'question']]

,id,question
72,73,What is a lambda function in Python?


In [7]:
df[df['question'].str.contains('comprehension', case=False)][['id', 'question']]

,id,question
88,89,What is list comprehension in Python?
89,90,What is dictionary comprehension in Python?
90,91,What is set comprehension in Python?
91,92,Is there a tuple comprehension in Python?
100,101,What is the difference between a list comprehe...


In [8]:
df[df['question'].str.contains('exception|try', case=False)][['id', 'question']]

,id,question
118,119,What is exception handling in Python?
119,120,What is the difference between try except else...
120,121,What are common built-in exceptions in Python?
121,122,How do you handle multiple exceptions in Python?
122,123,How do you raise an exception in Python?
123,124,What is a custom exception in Python?
124,125,What is the difference between a syntax error ...


In [9]:
THRESHOLD = 0.3

# (question, expected id). None means "should be rejected"
test_cases = [
    ("how to create a list from another list in one line", 89),
    ("difference between / and //", 20),
    ("what is a lambda", 73),
    ("how do I handle errors in python", 119),
    ("what is the capital of France", None),
    ("who won the world cup", None),
    ("best pizza recipe", None),
]

python_total = python_correct = 0
offtopic_total = offtopic_correct = 0

for question, expected_id in test_cases:
    vec = vectorizer.transform([preprocess(question)])
    scores = cosine_similarity(vec, tfidf_matrix).flatten()
    idx = scores.argmax()
    score = scores[idx]
    matched_id = df.iloc[idx]['id']

    if expected_id is None:
        offtopic_total += 1
        ok = score < THRESHOLD
        offtopic_correct += ok
    else:
        python_total += 1
        ok = (matched_id == expected_id) and (score >= THRESHOLD)
        python_correct += ok

    print(f"{'PASS' if ok else 'FAIL'} | score={score:.2f} | matched id={matched_id} | {question}")

print(f"\nPython questions: {python_correct}/{python_total}")
print(f"Off-topic rejected: {offtopic_correct}/{offtopic_total}")

FAIL | score=0.46 | matched id=43 | how to create a list from another list in one line
PASS | score=0.88 | matched id=20 | difference between / and //
PASS | score=0.47 | matched id=73 | what is a lambda
FAIL | score=0.33 | matched id=122 | how do I handle errors in python
PASS | score=0.00 | matched id=1 | what is the capital of France
PASS | score=0.00 | matched id=1 | who won the world cup
PASS | score=0.00 | matched id=1 | best pizza recipe

Python questions: 2/4
Off-topic rejected: 3/3


In [10]:
extra_keywords = {
    89: "create list from another list one line shortcut loop compact",
    119: "errors error handling try catch bug crash",
}

df['keywords'] = df['id'].map(extra_keywords).fillna('')
df['search_text'] = (df['question'] + ' ' + df['keywords']).apply(preprocess)

vectorizer2 = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix2 = vectorizer2.fit_transform(df['search_text'])

for q in ["how to create a list from another list in one line",
          "how do I handle errors in python"]:
    vec = vectorizer2.transform([preprocess(q)])
    scores = cosine_similarity(vec, tfidf_matrix2).flatten()
    idx = scores.argmax()
    print(q, "->", df.iloc[idx]['question'], f"(id={df.iloc[idx]['id']}, score={scores[idx]:.2f})")

how to create a list from another list in one line -> What is list comprehension in Python? (id=89, score=0.72)
how do I handle errors in python -> How do you handle multiple exceptions in Python? (id=122, score=0.34)


In [11]:
extra_keywords = {
    89: "create list from another list one line shortcut loop compact",
    119: "handle handle error handle errors error errors try catch bug crash",
}

df['keywords'] = df['id'].map(extra_keywords).fillna('')
df['search_text'] = (df['question'] + ' ' + df['keywords']).apply(preprocess)

vectorizer2 = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix2 = vectorizer2.fit_transform(df['search_text'])

for q in ["how to create a list from another list in one line",
          "how do I handle errors in python"]:
    vec = vectorizer2.transform([preprocess(q)])
    scores = cosine_similarity(vec, tfidf_matrix2).flatten()
    idx = scores.argmax()
    print(q, "->", df.iloc[idx]['question'], f"(id={df.iloc[idx]['id']}, score={scores[idx]:.2f})")

how to create a list from another list in one line -> What is list comprehension in Python? (id=89, score=0.72)
how do I handle errors in python -> What is exception handling in Python? (id=119, score=0.72)


In [12]:
THRESHOLD = 0.3

# New questions, NOT used for tuning. Expected ids are placeholders (0) for now.
fresh_tests = [
    "how do I define a function",
    "what is a dictionary",
    "how to open and read a file",
    "what is the difference between a list and a tuple",
    "how do I reverse a string",
    "what is a decorator",
    "who is the president of india",
    "how to cook rice",
]

for q in fresh_tests:
    vec = vectorizer2.transform([preprocess(q)])
    scores = cosine_similarity(vec, tfidf_matrix2).flatten()
    idx = scores.argmax()
    print("Asked  :", q)
    print("Matched:", df.iloc[idx]['question'], f"(id={df.iloc[idx]['id']}, score={scores[idx]:.2f})")
    print("-" * 40)

Asked  : how do I define a function
Matched: What is a function in Python? (id=67, score=0.64)
----------------------------------------
Asked  : what is a dictionary
Matched: What is a dictionary in Python? (id=60, score=0.59)
----------------------------------------
Asked  : how to open and read a file
Matched: How do you read a file in Python? (id=127, score=0.82)
----------------------------------------
Asked  : what is the difference between a list and a tuple
Matched: What is the difference between a list and a tuple? (id=53, score=1.00)
----------------------------------------
Asked  : how do I reverse a string
Matched: How do you reverse a string in Python? (id=36, score=0.85)
----------------------------------------
Asked  : what is a decorator
Matched: What is a decorator in Python? (id=93, score=0.64)
----------------------------------------
Asked  : who is the president of india
Matched: What is Python? (id=1, score=0.00)
----------------------------------------
Asked  : how

In [13]:
df[df['question'].str.contains('function', case=False)][['id', 'question']].head(15)

,id,question
28,29,What is the range() function in Python?
66,67,What is a function in Python?
72,73,What is a lambda function in Python?
79,80,What are first-class functions in Python?
109,110,What is the super() function in Python?
141,142,What are common NumPy statistical functions?
153,154,What is the apply function in Pandas?


In [14]:
real = ["difference between / and //", "what is a lambda", "how do I define a function",
        "what is a dictionary", "how to open and read a file", "what is a decorator"]
off = ["what is the capital of France", "who is the president of india",
       "who won the world cup", "how to cook rice", "what is the tallest mountain"]

def top_score(q):
    vec = vectorizer2.transform([preprocess(q)])
    return cosine_similarity(vec, tfidf_matrix2).max()

print("REAL questions")
for q in real: print(f"  {top_score(q):.2f}  {q}")
print("OFF-TOPIC questions")
for q in off: print(f"  {top_score(q):.2f}  {q}")

REAL questions
  0.90  difference between / and //
  0.55  what is a lambda
  0.64  how do I define a function
  0.59  what is a dictionary
  0.82  how to open and read a file
  0.64  what is a decorator
OFF-TOPIC questions
  0.00  what is the capital of France
  0.00  who is the president of india
  0.00  who won the world cup
  0.00  how to cook rice
  0.00  what is the tallest mountain


In [15]:
import preprocess as p
p.STOP.add('is')   # temporary: treat "is" as a stop word for this session

# Rebuild the improved model with the new preprocessing
df['search_text'] = (df['question'] + ' ' + df['keywords']).apply(preprocess)
vectorizer2 = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix2 = vectorizer2.fit_transform(df['search_text'])

real2 = real + ["What does the // operator do?", "what is python",
                "what is the difference between is and =="]

print("REAL questions")
for q in real2: print(f"  {top_score(q):.2f}  {q}")
print("OFF-TOPIC questions")
for q in off: print(f"  {top_score(q):.2f}  {q}")

REAL questions
  0.90  difference between / and //
  0.55  what is a lambda
  0.64  how do I define a function
  0.59  what is a dictionary
  0.82  how to open and read a file
  0.64  what is a decorator
  0.31  What does the // operator do?
  1.00  what is python
  0.50  what is the difference between is and ==
OFF-TOPIC questions
  0.00  what is the capital of France
  0.00  who is the president of india
  0.00  who won the world cup
  0.00  how to cook rice
  0.00  what is the tallest mountain
